# 🔍 Consultor de Jogos — Copa 2026

Digite o confronto no formato **`BRASIL X ARGENTINA`** e veja as análises completas do modelo.

---

In [ ]:
from pathlib import Path
import re
import unicodedata
import warnings

import ipywidgets as widgets
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110

DATA_DIR = Path('../data')
df      = pd.read_csv(DATA_DIR / 'probs_fase_de_grupos.csv')
df_sum  = pd.read_csv(DATA_DIR / 'summary.csv')

# Score columns: (home_goals, away_goals)
_WORDS = ['zero', 'one', 'two', 'three', 'four']
SCORE_COLS = {f'{a}_{b}': (i, j) for i, a in enumerate(_WORDS) for j, b in enumerate(_WORDS)}

ALL_TEAMS = sorted(set(df['home_team']) | set(df['away_team']))

def _norm(s: str) -> str:
    s = s.strip().lower()
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()

_NORM_MAP = {_norm(t): t for t in ALL_TEAMS}

def _find_team(query: str) -> str | None:
    q = _norm(query)
    if q in _NORM_MAP:
        return _NORM_MAP[q]
    for norm, orig in _NORM_MAP.items():
        if q in norm or norm.startswith(q[:4]):
            return orig
    return None

def _parse(query: str):
    parts = re.split(r'\s+[Xx×]\s+', query.strip())
    return (parts[0].strip(), parts[1].strip()) if len(parts) == 2 else (None, None)


def show_match(query: str) -> None:
    raw1, raw2 = _parse(query)
    if not raw1:
        print('❌  Formato inválido — use: BRASIL X ARGENTINA')
        return

    t1, t2 = _find_team(raw1), _find_team(raw2)
    if not t1:
        print(f'❌  Time não encontrado: "{raw1}"')
        print(f'    Times disponíveis: {', '.join(ALL_TEAMS[:10])} ...')
        return
    if not t2:
        print(f'❌  Time não encontrado: "{raw2}"')
        return

    mask = (
        ((df['home_team'] == t1) & (df['away_team'] == t2))
        | ((df['home_team'] == t2) & (df['away_team'] == t1))
    )
    rows = df[mask]
    if rows.empty:
        print(f'⚠️   Jogo {t1} × {t2} não encontrado na fase de grupos.')
        return

    row   = rows.iloc[0]
    home  = row['home_team']
    away  = row['away_team']
    p_h   = row['home_win']
    p_d   = row['draw']
    p_a   = row['away_win']

    # Most-probable score + xG
    best  = max(SCORE_COLS, key=lambda c: row[c])
    bh, ba = SCORE_COLS[best]
    tot   = sum(row[c] for c in SCORE_COLS)
    xg_h  = sum(SCORE_COLS[c][0] * row[c] for c in SCORE_COLS) / tot if tot else 0
    xg_a  = sum(SCORE_COLS[c][1] * row[c] for c in SCORE_COLS) / tot if tot else 0

    sh = df_sum[df_sum['team'] == home]
    sa = df_sum[df_sum['team'] == away]

    # ── Figure ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(14, 10))
    fig.suptitle(
        f"{home}  ×  {away}    |    Grupo {row['group']}  •  {row['date']}",
        fontsize=15, fontweight='bold', y=0.995,
    )
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.48, wspace=0.35)

    # 1. V / E / D
    ax1 = fig.add_subplot(gs[0, 0])
    vals = [p_h, p_d, p_a]
    lbls = [f'{home}\n(Casa)', 'Empate', f'{away}\n(Fora)']
    clrs = ['#27ae60', '#95a5a6', '#c0392b']
    bars = ax1.barh(lbls, vals, color=clrs, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax1.text(
            bar.get_width() + 0.4, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontweight='bold', fontsize=12,
        )
    ax1.set_xlim(0, max(vals) * 1.35)
    ax1.set_xlabel('Probabilidade (%)')
    ax1.set_title('Vitória / Empate / Derrota', fontweight='bold')
    ax1.spines[['top', 'right']].set_visible(False)

    # 2. Score card
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.axis('off')
    ax2.set_title('Placar mais provável + xG', fontweight='bold')
    ax2.text(0.5, 0.8, f'{bh}  —  {ba}', ha='center', fontsize=48,
             fontweight='bold', transform=ax2.transAxes, color='#2c3e50')
    ax2.text(0.5, 0.62, 'gols Casa  —  gols Fora', ha='center',
             fontsize=9, color='#7f8c8d', transform=ax2.transAxes)
    for x_pos, label, val, color in [
        (0.22, f'xG\n{home}', xg_h, '#27ae60'),
        (0.78, f'xG\n{away}', xg_a, '#c0392b'),
    ]:
        ax2.text(x_pos, 0.38, label, ha='center', fontsize=9,
                 color='#7f8c8d', transform=ax2.transAxes)
        ax2.text(x_pos, 0.16, f'{val:.2f}', ha='center', fontsize=30,
                 fontweight='bold', color=color, transform=ax2.transAxes)

    # 3. Score heatmap
    ax3 = fig.add_subplot(gs[1, 0])
    matrix = np.zeros((5, 5))
    for col, (h, a) in SCORE_COLS.items():
        if h < 5 and a < 5:
            matrix[a][h] = row[col]   # row=away goals, col=home goals
    total_m = matrix.sum()
    pct = matrix / total_m * 100 if total_m > 0 else matrix

    ax3.imshow(pct, cmap='YlOrRd', aspect='auto')
    ax3.set_xticks(range(5))
    ax3.set_yticks(range(5))
    ax3.set_xticklabels(range(5))
    ax3.set_yticklabels(range(5))
    ax3.set_xlabel(f'Gols {home} (Casa)', fontweight='bold')
    ax3.set_ylabel(f'Gols {away} (Fora)', fontweight='bold')
    ax3.set_title('Heatmap de Placares (%)', fontweight='bold')

    mx = pct.max()
    for i in range(5):
        for j in range(5):
            v = pct[i][j]
            txt_c = 'white' if v > mx * 0.55 else '#2c3e50'
            ax3.text(j, i, f'{v:.1f}', ha='center', va='center',
                     fontsize=8, color=txt_c)

    # 4. Tournament progression
    ax4 = fig.add_subplot(gs[1, 1])
    phases   = ['round_of_32', 'round_of_16', 'quarterfinals',
                'semifinals', 'final', 'champion']
    lbls_pt  = ['R32', 'Oitavas', 'Quartas', 'Semi', 'Final', 'Campeão']
    x = np.arange(len(phases))
    w = 0.35

    if not sh.empty and not sa.empty:
        v_h = [sh.iloc[0][p] for p in phases]
        v_a = [sa.iloc[0][p] for p in phases]
        ax4.bar(x - w / 2, v_h, w, label=home, color='#3498db', alpha=0.85)
        ax4.bar(x + w / 2, v_a, w, label=away, color='#e67e22', alpha=0.85)
        ax4.set_xticks(x)
        ax4.set_xticklabels(lbls_pt, fontsize=9)
        ax4.set_ylabel('Probabilidade (%)')
        ax4.set_title('Chance de Passar por Fase', fontweight='bold')
        ax4.legend(fontsize=9)
        ax4.spines[['top', 'right']].set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

print('✅  Pronto! Execute a célula abaixo para abrir o consultor.')

In [ ]:
txt = widgets.Text(
    placeholder='Ex: BRASIL X ARGENTINA',
    layout=widgets.Layout(width='320px'),
    style={'description_width': '0px'},
)
btn = widgets.Button(
    description='Analisar',
    button_style='primary',
    layout=widgets.Layout(width='110px'),
)
out = widgets.Output()

def _run(_=None):
    with out:
        clear_output(wait=True)
        show_match(txt.value)

btn.on_click(_run)
txt.on_submit(_run)   # Enter também funciona

display(widgets.HBox([txt, btn]), out)